## 深度学习与空间智能 第二次作业

任务2：场景目标检测与视频多目标跟踪

### 任务（1）训练模型

使用 Road Vehicle Images Dataset 数据集，微调训练 YOLOv8 或同类现代单阶段模型，得到你的专属检测模型。

In [1]:
import kagglehub

# Download latest version
# 下载 road-vehicle-images-dataset 数据集
path = kagglehub.dataset_download("ashfakyeafi/road-vehicle-images-dataset")
print("Path to dataset files:", path)

D:\Study\Courses\2026Spring\深度学习\hw2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\17961\.cache\kagglehub\datasets\ashfakyeafi\road-vehicle-images-dataset\versions\2


接下来，加载预训练模型，并开始训练。

In [1]:
# # 安装 PyTorch (请根据你的显卡型号去 pytorch.org 查找对应命令，以下是示例)
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
#
# # 安装 Ultralytics (包含 YOLOv8)
# pip install ultralytics

from ultralytics import YOLO

# 1. 加载预训练模型
# 'yolov8n.pt' 是 Nano 版本，速度最快但精度稍低。
# 也可以尝试 'yolov8s.pt' (Small) 或 'yolov8m.pt' (Medium)。
# 如果还没下载预训练权重，代码会自动从网上下载。
model = YOLO('yolov8n.pt')

# 2. 开始训练
# 参数说明：
# data: 你的 data.yaml 路径
# epochs: 训练轮数，一般建议 50-100
# imgsz: 图片尺寸，默认 640
# batch: 批大小，根据显存调整，-1 表示自动调整
# device: 0 表示使用 GPU，'cpu' 表示使用 CPU
results = model.train(
    data='./trafic_data/data_1.yaml',
    epochs=1,
    imgsz=640,
    batch=16,
    # device=0,
    device='cpu'
)

New https://pypi.org/project/ultralytics/8.4.50 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.45  Python-3.14.0 torch-2.11.0+cpu CPU (Intel Core Ultra 5 225H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./trafic_data/data_1.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, n

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Study\\Courses\\2026Spring\\深度学习\\hw2\\trafic_data\\train\\images\\Asraf_24_jpg.rf.9487899755918da526e30b26f9d57f24.jpg'

### 任务（2）：视频流检测与多目标跟踪

准备一段时长 10-30 秒的测试视频（可以是自己手
机拍摄的校园/路口视频等）。利用你训练好的模型，结合多目标跟踪算法（如
YOLOv8 内置的 tracking），对视频进行逐帧推理。要求不仅输出 Bounding Box
与类别，还必须为每个同一目标分配稳定的跟踪 ID（Tracking ID）。


In [5]:
import cv2

video_path = "test_video.mp4"  # 替换为你的实际视频路径
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("无法打开视频文件，请检查路径是否正确！")
else:
    # 获取视频的宽度、高度和帧率
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    print(f"🎥 原始视频信息:")
    print(f"   - 宽度 (Width): {frame_width} px")
    print(f"   - 高度 (Height): {frame_height} px")
    print(f"   - 帧率 (FPS): {fps}")
    print(f"   - 分辨率: {frame_width}x{frame_height}")

    cap.release()

🎥 原始视频信息:
   - 宽度 (Width): 852 px
   - 高度 (Height): 480 px
   - 帧率 (FPS): 29
   - 分辨率: 852x480


接下来处理视频，进行目标的识别和跟踪。

In [5]:
import cv2
from ultralytics import YOLO

# 1. 加载模型
model_path = 'runs/detect/train/weights/best.pt'
model = YOLO(model_path)

# 2. 视频输入与输出设置
video_path = "test_video_cc.mp4"       # 测试视频路径
output_path = "output_tracked.mp4"  # 输出视频路径

cap = cv2.VideoCapture(video_path)

# 获取视频的宽、高和帧率，用于创建写入对象
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# 定义视频编码器和创建 VideoWriter 对象
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

print("开始处理视频...")

# 3. 逐帧处理
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # --- 核心推理部分 ---
    # track: 开启跟踪模式
    # persist: 在多帧之间保持跟踪状态
    # tracker: 指定跟踪算法配置文件 (可选: bytetrack.yaml 或 botsort.yaml)
    # conf: 置信度阈值，根据训练结果，设置 0.359
    results = model.track(frame, persist=True, tracker="bytetrack.yaml", conf=0.359)

    # # 4. 可视化结果
    # # plot() 函数会自动绘制 BBox, ID 和 类别
    # # 如果你想要自定义绘制，可以遍历 results[0].boxes 获取数据
    # annotated_frame = results[0].plot()
    #
    # # 写入视频文件
    # out.write(annotated_frame)

    # --- 4. 自定义可视化结果（只显示 ID，不显示类别）---
    annotated_frame = frame.copy() # 复制原始帧用于绘制

    # 检查当前帧是否有检测到目标并分配了ID
    if results[0].boxes.id is not None:
        # 提取边界框坐标、跟踪ID
        boxes = results[0].boxes.xyxy.cpu().numpy()  # xyxy格式的坐标
        track_ids = results[0].boxes.id.int().cpu().numpy() # 跟踪ID

        for box, track_id in zip(boxes, track_ids):
            x1, y1, x2, y2 = map(int, box) # 获取左上角和右下角坐标

            # 1. 绘制边界框 (绿色框，线宽为2)
            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # 2. 绘制仅包含 ID 的文本标签
            label_text = f"ID: {track_id}"
            # 在边界框上方添加文字背景，防止文字看不清
            (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)
            cv2.rectangle(annotated_frame, (x1, y1 - text_h - 10), (x1 + text_w, y1), (0, 255, 0), -1)
            # 绘制黑色文字
            cv2.putText(annotated_frame, label_text, (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 2)

    out.write(frame)

# 5. 释放资源
cap.release()
out.release()
cv2.destroyAllWindows()
print(f"视频处理完成，已保存至: {output_path}")

开始处理视频...

0: 384x640 16 cars, 28.9ms
Speed: 2.5ms preprocess, 28.9ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 16 cars, 25.0ms
Speed: 1.2ms preprocess, 25.0ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 17 cars, 23.9ms
Speed: 1.1ms preprocess, 23.9ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 15 cars, 22.8ms
Speed: 1.1ms preprocess, 22.8ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 15 cars, 25.9ms
Speed: 1.1ms preprocess, 25.9ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 cars, 24.6ms
Speed: 1.6ms preprocess, 24.6ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 15 cars, 23.8ms
Speed: 1.6ms preprocess, 23.8ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 17 cars, 22.2ms
Speed: 1.3ms preprocess, 22.2ms inference, 0.6ms postprocess per image at shape (1, 3, 

### 任务（3）遮挡与跳变分析

多目标跟踪中最具挑战性的问题是物体遮挡。请在你的测试视频中，挑选一个发生目标遮挡或密集交汇的片段，截取连续的 3-4 帧画面进行可视化展示。结合课上所学理论，详细分析跟踪算法在面对此场景时，是成功维持了原有 ID，还是发生了目标丢失/ID 跳变，并简述其原理或原因。


分析内容见本次作业报告。

### 任务（4）越线计数

在视频画面中设定一条虚拟的线，利用检测框中心的坐标与 Tracking ID 的连续性，编写简单的逻辑判断，统计并显示视频播放期间“跨越该线的物体总数”。

In [1]:
import cv2
from ultralytics import YOLO

# --- 配置部分 ---
model_path = 'runs/detect/train/weights/best.pt'  # 你的训练权重
video_path = "test_video_cc.mp4"                     # 输入视频
output_path = "output_counted.mp4"                # 输出视频

# --- 初始化 ---
model = YOLO(model_path)
cap = cv2.VideoCapture(video_path)

# 获取视频信息
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# 虚拟线的坐标 (x1, y1, x2, y2)
# 这里设定为画面中间的一条水平线
line_x = w/2
line_y = h/2
line_start = (int(w/2), 0)
line_end = (int(w/2), h) # 假设宽度足够，或者在循环中动态获取宽度

out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

# 计数相关变量
total_count = 0
crossed_ids = set()  # 用来存储已经跨过线的物体ID，防止重复计数

print("开始越线计数推理...")
# 在 while cap.isOpened() 循环之前，初始化一个字典来存储历史位置
track_history = {}

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # --- YOLOv8 推理与追踪 ---
    results = model.track(frame, persist=True, conf=0.3)

    if results[0].boxes is not None and results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy()

        for box, track_id, cls in zip(boxes, track_ids, classes):
            x1, y1, x2, y2 = box
            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)

            track_id = int(track_id)  # 确保 track_id 是整数

            # --- 优化后的越线判断逻辑 ---
            # 1. 获取该 ID 上一帧的中心点 X 坐标 (如果没有则为 None)
            prev_cx = track_history.get(track_id)

            # 2. 判断是否发生跨越 (从左往右穿过 line_x)
            # 条件：上一帧在左边 (或没有历史记录) AND 这一帧到了右边
            if prev_cx is not None and prev_cx <= line_x and cx > line_x:
                total_count += 1
                print(f"检测到 ID {track_id} 跨越中线！当前总数: {total_count}")

            # 3. 更新该 ID 的最新位置
            track_history[track_id] = cx

            # --- 绘图部分 (保持不变) ---
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
            # label = f"ID: {track_id}"
            label = f"ID: {track_id} {model.names[int(cls)]}"
            # 指定你想要的 BGR 颜色，例如深蓝色 (150, 50, 0)
            my_color = (255, 255, 255)

            # fontScale=0.4 让字号变小，my_color 是你自定义的颜色
            cv2.putText(frame, label, (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, my_color, 2)
            # cv2.putText(frame, label, (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    # --- 绘制虚拟线和计数 ---
    cv2.line(frame, line_start, line_end, (0, 255, 255), 3)
    cv2.putText(frame, f"Count: {total_count}", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 4)

    out.write(frame)

cap.release()
out.release()
cv2.destroyAllWindows()
print(f"处理完成！总计数: {total_count}")

开始越线计数推理...

0: 384x640 17 cars, 51.7ms
Speed: 65.2ms preprocess, 51.7ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 17 cars, 31.0ms
Speed: 1.9ms preprocess, 31.0ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 17 cars, 33.2ms
Speed: 1.2ms preprocess, 33.2ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 17 cars, 34.0ms
Speed: 1.5ms preprocess, 34.0ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 17 cars, 29.3ms
Speed: 1.0ms preprocess, 29.3ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 15 cars, 34.8ms
Speed: 1.1ms preprocess, 34.8ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 17 cars, 30.4ms
Speed: 1.0ms preprocess, 30.4ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 18 cars, 32.8ms
Speed: 1.0ms preprocess, 32.8ms inference, 0.9ms postprocess per image at shape (1, 